# 08 Experiment Tracking with MLflow
This notebook demonstrates how to track training runs, parameters, and metrics using MLflow. This is essential for comparing different model versions and hyperparameter experiments.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import mlflow
import mlflow.pytorch
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

from pneumonia_classifier.ml.model.arch import Net

## Configuration
Set the device, data paths, and initialize the MLflow experiment.

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = r'j:\Users\ayush\Desktop\code\pneumonia_classifier\artifacts\02_12_2025_08_52_04\data_ingestion\data\data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
INPUT_SIZE = (224, 224)

# MLflow Setup
mlflow.set_experiment("pneumonia_classifier_experiment")

2026/02/28 16:46:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/28 16:46:22 INFO mlflow.store.db.utils: Updating database tables
2026/02/28 16:47:03 INFO mlflow.tracking.fluent: Experiment with name 'pneumonia_classifier_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///j:/Users/ayush/Desktop/code/pneumonia_classifier/notebooks/mlruns/1', creation_time=1772277423162, experiment_id='1', last_update_time=1772277423162, lifecycle_stage='active', name='pneumonia_classifier_experiment', tags={}, workspace='default'>

## Training Function with MLflow Logging
We wrap the training and evaluation logic in an MLflow run to capture everything.

In [3]:
def train_with_tracking(lr=0.001, batch_size=16, epochs=5):
    with mlflow.start_run():
        # Log Parameters
        mlflow.log_param("lr", lr)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("epochs", epochs)
        
        # Data Loading
        transform = transforms.Compose([
            transforms.Resize(INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
        test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        model = Net().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        # Enable automatic logging for PyTorch
        # mlflow.pytorch.autolog()
        
        for epoch in range(1, epochs + 1):
            # Training
            model.train()
            total_loss = 0
            for data, target in train_loader:
                data, target = data.to(DEVICE), target.to(DEVICE)
                optimizer.zero_grad()
                output = model(data)
                loss = F.nll_loss(output, target)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            avg_loss = total_loss / len(train_loader)
            mlflow.log_metric("train_loss", avg_loss, step=epoch)
            
            # Evaluation
            model.eval()
            all_preds = []
            all_targets = []
            with torch.no_grad():
                for data, target in test_loader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    output = model(data)
                    pred = output.argmax(dim=1).cpu().numpy()
                    all_preds.extend(pred)
                    all_targets.extend(target.cpu().numpy())
            
            # Calculate expanded metrics (since Recall is already 1.0)
            precision = precision_score(all_targets, all_preds, pos_label=1)
            recall = recall_score(all_targets, all_preds, pos_label=1)
            f1 = f1_score(all_targets, all_preds, pos_label=1)
            
            mlflow.log_metric("precision", precision, step=epoch)
            mlflow.log_metric("recall", recall, step=epoch)
            mlflow.log_metric("f1_score", f1, step=epoch)
            
            print(f"Epoch {epoch}: Loss {avg_loss:.4f}, Precision {precision:.4f}, Recall {recall:.4f}, F1 {f1:.4f}")
        
        # Log the final model
        mlflow.pytorch.log_model(model, "model")
        
        print(f"Run finished. View results in MLflow UI by running 'mlflow ui' in your terminal.")

## Run Tracking Experiments
Run experiments with different learning rates to see how they affect precision and recall.

In [4]:
# Run a few experiments
train_with_tracking(lr=0.0001, batch_size=8, epochs=3)
train_with_tracking(lr=0.001, batch_size=16, epochs=3)

Epoch 1: Loss 0.6119, Precision 0.5263, Recall 1.0000, F1 0.6897
Epoch 2: Loss 0.4679, Precision 1.0000, Recall 0.8000, F1 0.8889


2026/02/28 16:50:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3: Loss 0.3964, Precision 1.0000, Recall 0.8333, F1 0.9091


2026/02/28 16:50:56 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/02/28 16:54:32 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\user\AppData\Local\Temp\tmp_swc1buy\model\data, flavor: pytorch). Fall back to return ['torch==2.10.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run finished. View results in MLflow UI by running 'mlflow ui' in your terminal.
Epoch 1: Loss 0.4456, Precision 0.5135, Recall 0.6333, F1 0.5672
Epoch 2: Loss 0.2787, Precision 0.5472, Recall 0.9667, F1 0.6988


2026/02/28 16:57:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3: Loss 0.1873, Precision 0.9667, Recall 0.9667, F1 0.9667


2026/02/28 16:57:17 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Run finished. View results in MLflow UI by running 'mlflow ui' in your terminal.


## Conclusion
Check the `mlruns` directory or start the MLflow UI to compare these runs. You'll likely see that although Recall hits 1.0 quickly, Precision and F1-score give you a better picture of the true performance.